In [17]:
from search import *
import random

class QueensWithValue(NQueensProblem):
    def value(self, state):
        n = len(state)
        non_attacking = 0

        for i in range(n):
            for j in range(i + 1, n):
                if state[i] != state[j] and abs(state[i] - state[j]) != abs(i - j):
                    non_attacking += 1

        return non_attacking


def run_queens_experiments(num_trials=100):
    results = {
        "steepest": 0,
        "random_restart": 0,
        "annealing": 0
    }

    for _ in range(num_trials):

        # Steepest
        problem = QueensWithValue(8)
        result = hill_climbing(problem)
        if problem.goal_test(result):
            results["steepest"] += 1

        # Annealing
        problem = QueensWithValue(8)
        result = simulated_annealing(problem)
        if problem.goal_test(result):
            results["annealing"] += 1

        # Random restart
        solved = False
        for _ in range(10):
            problem = QueensWithValue(8)
            result = hill_climbing(problem)
            if problem.goal_test(result):
                solved = True
                break

        if solved:
            results["random_restart"] += 1

    return results

In [18]:
class PuzzleWithValue(EightPuzzle):
    def value(self, state):
        distance = 0
        goal = (1,2,3,4,5,6,7,8,0)

        for i in range(9):
            if state[i] == 0:
                continue

            goal_index = goal.index(state[i])
            x1, y1 = divmod(i, 3)
            x2, y2 = divmod(goal_index, 3)

            distance += abs(x1 - x2) + abs(y1 - y2)

        return -distance


def random_puzzle(moves=50):
    problem = EightPuzzle((1,2,3,4,5,6,7,8,0))
    state = problem.initial

    for _ in range(moves):
        actions = problem.actions(state)
        state = problem.result(state, random.choice(actions))

    return state


def get_optimal_cost(state):
    problem = EightPuzzle(state)
    result = astar_search(problem)
    if result is None:
        return None
    return len(result.solution())


def run_puzzle_experiments(num_trials=50):
    results = []

    for _ in range(num_trials):
        state = random_puzzle()

        optimal_cost = get_optimal_cost(state)
        if optimal_cost is None:
            continue

        # Hill Climbing
        problem = PuzzleWithValue(state)
        hc_result = hill_climbing(problem)
        hc_solved = problem.goal_test(hc_result)

        # Simulated Annealing
        problem = PuzzleWithValue(state)
        sa_result = simulated_annealing(problem)
        sa_solved = problem.goal_test(sa_result)

        # Random Restart
        rr_solved = False
        for _ in range(10):
            problem = PuzzleWithValue(state)
            rr_result = hill_climbing(problem)
            if problem.goal_test(rr_result):
                rr_solved = True
                break

        results.append({
            "hc_solved": hc_solved,
            "sa_solved": sa_solved,
            "rr_solved": rr_solved
        })

    return results

In [19]:
queens_results = run_queens_experiments(100)
puzzle_results = run_puzzle_experiments(50)


In [20]:
print("=== 8-Queens Results ===")
for k, v in queens_results.items():
    print(f"{k}: {v/100:.2f} success rate")

print("\n=== 8-Puzzle Results ===")

hc_success = sum(1 for r in puzzle_results if r["hc_solved"]) / len(puzzle_results)
sa_success = sum(1 for r in puzzle_results if r["sa_solved"]) / len(puzzle_results)
rr_success = sum(1 for r in puzzle_results if r["rr_solved"]) / len(puzzle_results)

print(f"Hill Climbing success: {hc_success:.2f}")
print(f"Simulated Annealing success: {sa_success:.2f}")
print(f"Random Restart success: {rr_success:.2f}")

=== 8-Queens Results ===
steepest: 0.00 success rate
random_restart: 0.00 success rate
annealing: 0.13 success rate

=== 8-Puzzle Results ===
Hill Climbing success: 0.36
Simulated Annealing success: 0.00
Random Restart success: 0.48
